# Threshold Method

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks

This method finds a threshold value based on the climatological median for each pixel and then compares the daily data to threshold. Values above the threshold are considered blooms and values below are not blooms. This step is testing different threshold values to determine what threshold most accurately models phytoplankton blooms on the NES. For testing, we chose 5-30% because these are values commonly used in bloom phenology literature. We then test these different threshold values on the daily data throughout the time series to determine which threshold is best for our region and for identifying the bloom metrics in question. 

This section includes functions to find the threshold values based on the climatology, create a mask to only include bloom values in an array, and function to clip data to smaller regions for different spatial comparisons to the mean. It culminates in a chosen threshold percentage for the entire NES region.

### Function for determining the climatological threshold value
Based on the climatological median, this function finds a certain percentage of that median and adds it to the median to get a threshold value for determining phytoplankton blooms. After inputing the annual climatology for the region, the median chlorophyll-a value for the annual climatology is extracted. The percent of the median defined by the threshold percentage (5%, 10%, 15%, etc.) is then added to the median to create the median threshold values. The threshold_value() function computes this number based on the threshold percentage specified. 
* Variables to include:
    * Threshold percentage = thld (thld=0.1 by default)
    * Path to climatology files = path (set to NES Annual Climatology 1997 - 2020 by default)

#### Plotting the Climatological Threshold
To begin visualizing the data, we plot the climatological median threshold value on a map to see the spatial differences across the region. Notably, the Georges Bank area and estuaries typically have a higher median threshold value, while the open ocean has a relatively low threshold value. This means that blooms are more likely to be detected in open ocean regions since Georges Bank and estuaries have naturally higher primary productivity.

### Create a mask to filter data for bloom conditions
These functions create a mask to show only pixels that exceed the threshold values set by the threshold_value() function above. When bloom_mask_numeric is plotted, a daily map will show regions under the threshold value as white (no values) and the rest of the values as their raw value.

This function (bloom_mask_Boolean) takes the input data and determines if values exceed the climatological threshold determined above (or found via the threshold_value function). If the value is less than or equal to the threshold, then it is reported as false. If it is greater than the threshold, it is reported as true. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include
    * Path to files = path (set to D8 NES shelf files by default)

This function (bloom_mask_numeric) functions similar to bloom_mask_Boolean, except it reports false values as 0 and true values retain their actual value. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include:
    * Path to files = path (set to D8 NES shelf files by default)

#### Mask Intervals
Finding the climatological threshold at a few different inteverals (5%, 10%, 15%, 20%, 25%, and 30%).

Plotting all of the masks (5% - 30%)
<br> Must pick a day of the year from 0 to 10,253

## Histograms of Data

### Histograms of specific areas within the regions

This step was done to investigate a randomly chosen 1 degree box and see how much of the chlorophyll-a data laid above the median threshold in question. Once the histograms were plotted for the actual chlorophyll-a values, lines representing that area's climatological median and calculated threshold values were overlaid to see how much data fell on either side of the thresholds. This was a higher spatial resolution method to begin testing the different threshold methods.

Function to clip the median chl-a data for a specific longitude and latitude
* Must input 
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

Function to clip the climatological data to the area 
* Must include
    * Path to file: path =
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

Function builds a square (polygon) of the area to plot
* Must include
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

Assign variables to the various datasets for plotting

#### Plotting

#### Determining the percentage of datapoints that lie above each threshold
Using the variables defined above for plotting, compare each value in the data set to that area's climatological median threshold.
* Threshold options:
    * clim_med = climatological median of the area (must use index of 0-2 in order of GOM, GB, MAB)
    * clim_5 = 5% above the climatological median (Must use indexing for all threshold values)
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2
* Dataset options:
    * ds_GOM = Gulf of Maine subset data
    * ds_GB = Georges Bank subset data
    * ds_MAB = Middle Atlantic Bight subset data

This was done to determine numerically the percentage of data that would be exceeding the threshold for each square of data. This helped to quantify what we could see on the previous histograms. It helped us eliminate 20%, 25%, and 30% from the list of viable thresholds. These thresholds excluded a large majority of the data in some regions and we wanted one threshold percentage for the whole NES region.

### Whole region analysis

We repeated the steps for the small 1-degree box analysis, this time spatially averaging the entire regions. In this step, the Gulf of Maine and Middle Atlantic Bight were separated into two regions each. This follows the shapefiles available but also covers spatial differences within the regions. From this analysis, we concluded that 5% was too low. To finalizae the decision between 10% and 15%, we plotted the threshold value on a time series to observe what qualitatively made sense. We chose 10% because we felt that it encompassed enough of the bloom data without excluding smaller peaks. You could however choose 15% as well, both seemed to do well at representing the general trends of the NES.

#### Creating the shapefiles for each region

Clip the data to the shapefiles

#### Plotting

#### NES Full Region

#### Calculating the percentage of points above the threshold
Using the variables defined above for plotting and percent_above_thld(), compare each value in the data set to that area's climatological median threshold.
* Threshold options: (Must use index to get the correct area)
    * clim_med = climatological median of the area
    * clim_5 = 5% above the climatological median
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2, 3, 4 (in the order of: Middle Atlantic Bight South, Middle Atlantic Bight North, Georges Bank, Gulf of Maine West, Gulf of Maine East)
* Dataset options:
    * MAB_south = Middle Atlantic Bight South data
    * MAB_north = Middle Atlantic Bight North data
    * GB_whole = Georges Bank data
    * GOM_west = Gulf of Maine West data
    * GOM_east = Gulf of Maine East data